# Reconstruction Demo

Данный ноутбук позволяет загрузить репозиторий и веса модели,

провести инференс на своём датасете, визуализировать несколько изображений и посчитать основные метрики

In [ ]:
repo_url = "https://github.com/cherrypy1/lensless-imaging" #репозиторий проекта

Здесь представлены основные переменные, которые можно менять

In [ ]:
#url к кастомному датасету на Google Drive
custom_zip_url = "https://drive.google.com/file/d/1djjcK1Izqlb6fdapTDMPTi0-0x9L_5vc/view?usp=sharing"  
#путь из корня, по которому сохраняются реконструкции
reconstruction_path = "data/saved/reconstructions"

#имя модели, доступные:
#  admm100  unrolled_admm20  leadmm5_pre_post  leadmm5_pre  leadmm5_post
model_name = "leadmm5_pre_post"

#количество картинок для генерации и подсчета метрик
count = 50

Код для корректной загрузки репозитория и установки зависимостей, также и в случаи перезапуска ячейки

In [ ]:
%cd /content
!rm -rf lensless_project
!git clone {repo_url} lensless_project
%cd lensless_project
!pip install -r requirements.txt

Загрузка финального чекпоинта для обучаемой модели (после 20 эпох)

In [ ]:
if model_name != "admm100":
    !python src/scripts/download_checkpoint.py --name {model_name}

Загрузка и распаковка кастомного датасета.

zip должен содержать директории `lensless/`, `masks/`, и опционально`lensed/`.

In [ ]:
!mkdir -p data/custom
!gdown --fuzzy {custom_zip_url} -O data/custom.zip
!unzip -q data/custom.zip -d data/custom
!find data/custom 

Запуск инференса модели по указанным параметрам

In [ ]:
from pathlib import Path

custom_root = Path("data/custom")

#admm100 не обучаема
if model_name != "admm100":
    checkpoint = f"checkpoints/{model_name}.pth"
else:
    checkpoint = "null"

#запуск инференса по указанным в начале параметрам
#count_time = True подсчитывает среднее время инференса. =False не считает
!python inference.py \
    model={model_name} \
    datasets.test.root={custom_root} \
    inferencer.from_pretrained={checkpoint} \
    inferencer.save_path={reconstruction_path} \
    inferencer.count_time=True  \
    inferencer.count={count}


Визуализация до 3 изображений.

Столбцы lensless/, reconstruction/ и lensed/ если есть

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

pred_root = Path(reconstruction_path)
#ids выбранных lensless картинок для визуализации
image_ids = [path.stem for path in sorted((custom_root / "lensless").glob("*.png"))[:min(count, 3)]]
#проверка наличия lensed в датасете
has_lensed = (custom_root / "lensed").exists()
titles = ["lensless", "reconstruction"]
if has_lensed:
    titles = ["lensed"] + titles

# создание графика через plt
fig, axes = plt.subplots(len(image_ids), len(titles), figsize=(4 * len(titles), 4 * len(image_ids)))
if len(image_ids) == 1:
    axes = [axes]

for row, image_id in zip(axes, image_ids):
    paths = [custom_root / "lensless" / f"{image_id}.png", pred_root / f"{image_id}.png"]
    if has_lensed:
        paths = [custom_root / "lensed" / f"{image_id}.png"] + paths
    for ax, path, title in zip(row, paths, titles):
        #Для симметричной визуализации
        image = Image.open(path).convert("RGB").resize((256, 256))
        ax.imshow(image)
        ax.set_title(f"{image_id}: {title}")
        ax.axis("off")
plt.tight_layout()

Подсчёт усреднённых по изображениям метрик PSNR, LPIPS, MSE, SSIM, 

если датасет содержит оригиналы изображений (lensed)

In [ ]:
from pathlib import Path

lensed_root = custom_root / "lensed"
recon_root = Path(reconstruction_path)
if lensed_root.exists():
    #скрипт для подсчёта метрик
    !python calculate_metrics.py --lensed {lensed_root} --reconstructions {recon_root} --count {count}